Librerias necesarias:

In [1]:
from astropy import units as u
from astropy.time import Time, TimeDelta
from astropy.coordinates import get_sun, SkyCoord, EarthLocation, Angle
import numpy as np
from astropy.table import Table

Definimos nuestros datos iniciales, que son editables.

In [2]:
# Cuerpo
ra = Angle('19h59m36s')
dec = Angle('22d43m16s')

# Observador (positivamente hacia el norte, este)
lat = Angle('28d18m00s')
lon = Angle('-16d30m35s')

# Fechas de observación (yyyy-mm-dd)
fecha_inicial = Time('2025-01-01')
fechas = fecha_inicial + TimeDelta(np.arange(365), format='jd')

print('Datos iniciales cargados')

Datos iniciales cargados


Definimos nuestras coordenadas, tanto del cuerpo como del observador y traemos el Sol

In [3]:
cuerpo = SkyCoord(ra=ra, dec=dec)
observador = EarthLocation(lat=lat, lon=lon)

print(f'El cuerpo celeste tiene ascensión recta {ra}, declinación {dec}')
print(f'El observador se encuentra a latitud {lat} y longitud {lat}')
print('Lista con los datos solares creada')
print(f'Se estudiará la visibilidad durante {len(fechas)} días')


El cuerpo celeste tiene ascensión recta 19.993333333333336 hourangle, declinación 22.72111111111111 deg
El observador se encuentra a latitud 28.3 deg y longitud 28.3 deg
Lista con los datos solares creada
Se estudiará la visibilidad durante 365 días


Tiempos sidéreos, orto y ocaso

In [4]:
def t_sid (obj, obs_lat):
    H_oc = (Angle(np.arccos(-np.tan(obj.dec)*np.tan(obs_lat)), unit=u.hourangle)).wrap_at('24h')
    H_or = (Angle(24, unit=u.hourangle)-H_oc).wrap_at('24h')

    sid_or = (H_or + obj.ra).wrap_at('24h')
    sid_oc = (H_oc + obj.ra).wrap_at('24h')

    return sid_or, sid_oc

Ventana de visibilidad:

In [5]:
def i_vis(noche, cuerpo_alto):
    def partir_int(intervalo):
      a1, a2 = intervalo
      if a1 < a2:
         return [(a1, a2)]
      else:
         return [(a1, Angle(24, unit=u.hourangle)), (Angle(0, unit=u.hourangle), a2)]

    def interseccion(i1, i2):
      a1, a2 = i1
      b1, b2 = i2
      inicio = max(a1, b1)
      fin = min(a2, b2)
      if inicio < fin:
         return inicio, fin
      return None

    N_partes = partir_int(noche)
    C_partes = partir_int(cuerpo_alto)

    visib = []
    for N in N_partes:
       for C in C_partes:
          inters = interseccion(N, C)
          if inters is not None:
             visib.append(inters)

    return visib

Paso a tiempos universales:

In [6]:
def TU (visib, fecha):
    TUs = []
    if (visib is None) or (len(visib) == 0):
        TU_0 = 0
        TU_1 = 0
        return []

    TS0 = fecha.sidereal_time('apparent', 'greenwich')

    for item in visib:

        TU_0 = (item[0] - observador.lon - TS0).wrap_at('24h')
        TU_1 = (item[1] - observador.lon - TS0).wrap_at('24h')
        TUs.append((TU_0, TU_1))

    return TUs

In [7]:
def duracion(TUs):
    suma=0
    for item in TUs:
        d = (item[1] - item[0]).wrap_at('24h').to_value(u.hourangle)
        suma += d
    return suma

In [8]:
results = Table(names=('Fecha', 'Intervalos de visibilidad', 'duración total'), dtype=('object', 'object', 'float64'))
sid_or_cuerpo, sid_oc_cuerpo = t_sid(cuerpo, observador.lat)

for fecha in fechas:
    sol = get_sun(fecha)

    sid_or_sol, sid_oc_sol = t_sid(sol, observador.lat)

    noche = (sid_oc_sol, sid_or_sol)
    cuerpo_alto = (sid_or_cuerpo, sid_oc_cuerpo)


    visib = i_vis(noche, cuerpo_alto)
    TUs = TU(visib, fecha)

    duration = duracion(TUs)

    TUs_decimales = [(str(round(t0.to_value(), 3)), str(round(t1.to_value(), 3))) for t0, t1 in TUs]

    results.add_row([fecha.strftime('%Y-%m-%d'), TUs_decimales, round(duration, 2)])

print(f'\nTotal de días calculados: {len(fechas)}')


Total de días calculados: 365


In [9]:
# Copiamos la tabla para no alterar la original
results2 = results.copy()

# Convertimos los intervalos de visibilidad a string para CSV
results2['Intervalos de visibilidad'] = [str(item) for item in results['Intervalos de visibilidad']]

# Guardamos
results2.write('visibilidad.csv', format='csv', overwrite=True)
print("Archivo 'visibilidad.csv' guardado correctamente.")


Archivo 'visibilidad.csv' guardado correctamente.
